In [3]:
import faiss
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from unsloth import FastLanguageModel
import torch

embedder = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index("medical_faiss.index")

with open("docs.json") as f:
    documents = json.load(f)

model_name = "../model/final_lora_weights"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    max_seq_length=4096,
    dtype=torch.bfloat16,
    load_in_4bit=True,
    device_map="auto"
)

FastLanguageModel.for_inference(model)

def query_rag(question, top_k=3):
    query_embedding = embedder.encode([question])
    distances, indices = index.search(np.array(query_embedding), top_k)
    retrieved_docs = [documents[i] for i in indices[0]]
    context = "\n\n".join(retrieved_docs)

    prompt = f"""
<s>[INST] You are a medical AI assistant. Answer strictly using the provided medical documents.
If the answer is not found, respond with "I don't know".

### Question:
{question}

### Relevant Documents:
{context}

### Answer:
[/INST]
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.15
        )

    ans = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return ans, retrieved_docs

==((====))==  Unsloth 2025.11.6: Fast Mistral patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Device does not support bfloat16. Will change to float16.
Unsloth: Will load ../model/final_lora_weights as a legacy tokenizer.


In [4]:
q = "What are the symptoms of dengue fever?"
answer, docs = query_rag(q)

print("\n===== Final Answer =====")
print(answer)

print("\n===== Retrieved Docs =====")
for d in docs:
    print("\n---\n", d)


===== Final Answer =====

 [INST] You are a medical AI assistant. Answer strictly using the provided medical documents.
If the answer is not found, respond with "I don't know".

### Question:
What are the symptoms of dengue fever?

### Relevant Documents:
Dengue. Dengue fever is a mosquito-borne disease caused by dengue virus, prevalent in tropical and subtropical areas. Most cases of dengue fever are either asymptomatic or manifest mild symptoms. Symptoms typically begin 3 to 14 days after infection. They may include a high fever, headache, vomiting, muscle and joint pains, and a characteristic skin itching and skin rash. Recovery generally takes two to seven days. In a small proportion of cases, the disease develops into severe dengue (previously known  ['As of July 2024, there is no specific antiviral treatment available for dengue fever.\nMost cases of dengue fever have mild symptoms, and recovery takes place in a few days. No treatment is required for these cases. Acetaminophen (